# STAT 764 · Lab 4 — The penalty is a decision too

**Assigned Tue Sep 29 · Due Tue Oct 6, 11:59 pm**

Individual work. Parts 1 and 2 need Meeting 7 (Tuesday Sep 29); Part 3 needs Meeting 8
(Thursday Oct 1), and question **(d)** needs the reading.

Same rules as before: `git pull`, copy into `work/` before editing, Restart &
Run All before submitting, upload the `.ipynb` to Canvas.

Lab 3 said the metric is a decision and the scale is a decision. This one adds two
more: **how hard you penalize**, and **how hard you search**. Neither shows up in the
number you report unless you go looking for it.

⏱ The whole lab runs in two to four minutes, depending on your laptop. Almost all of
that is Part 3, and that is normal.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:                       # you are inside (or just outside) your clone
    sys.path.insert(0, str(found[0] / "course"))
else:                           # Colab, or a copy saved outside the clone
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

from stat764 import load

In [ ]:
import warnings

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Pool_QC is blank for all but 13 of the 2,930 houses, so most small samples contain
# none. sklearn drops the empty column and prints a warning on EVERY fit that does.
# The message is harmless and would bury your output, so it is switched off -- that one
# message only.
warnings.filterwarnings("ignore", message="Skipping features without any observed values")

ames = load("ames.csv")
y = ames["SalePrice"]

# The WIDE design: every column except the two ID numbers and the answer.
NUMERIC = [c for c in ames.select_dtypes("number").columns
           if c not in ("Order", "PID", "SalePrice")]
CATEGORICAL = ames.select_dtypes(exclude="number").columns.tolist()
X = ames[NUMERIC + CATEGORICAL]


def wide_pipeline(model, scale=True):
    """The Meeting 2 skeleton, on every column. scale=False drops the StandardScaler."""
    numeric_steps = [("fill", SimpleImputer(strategy="median"))]
    if scale:
        numeric_steps.append(("scale", StandardScaler()))
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline(numeric_steps), NUMERIC),
            ("cat", Pipeline([("fill", SimpleImputer(strategy="constant",
                                                     fill_value="Missing")),
                              ("encode", OneHotEncoder(handle_unknown="ignore"))]),
             CATEGORICAL),
        ])),
        ("model", model),
    ])


width = wide_pipeline(LinearRegression()).fit(X, y).named_steps["prep"].transform(X).shape[1]
print(f"{len(ames)} sales · {len(NUMERIC)} numeric + {len(CATEGORICAL)} categorical columns"
      f" -> {width} columns once the categories are one-hot encoded")

Everything in this lab uses that one design and that one pipeline, so your numbers are
comparable with each other and with mine. Your job is the experiments, not the plumbing.

---

## Part 1 · The brake, and when it matters (15 points)

Fit three models on the wide design — `LinearRegression()`, `Ridge(alpha=30)` and
`Lasso(alpha=300, max_iter=20000)` — at two training sizes:

- **all of the training split** (2,197 houses)
- **its first 300 rows**

Score both on the same 733 held-out houses, and do it on **ten splits**, not one:

```python
for seed in range(10):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=733, random_state=seed)
    for n in (300, len(X_train)):
        ...    # fit all three on X_train.iloc[:n], y_train.iloc[:n]; score on X_test
```

Report one table: for each model at each size, the **median** and the **worst**
held-out R-squared over the ten splits, and the median MAE in dollars. Then count, at
each size, **on how many of the ten splits** each penalized model beat OLS.

Also report, for one split, OLS's R-squared **on its own 300 training houses**, and how
many columns the pipeline built from those 300 rows. You will want both for 4(a).

⚠ The alphas are fixed on purpose. Part 3 is where you choose them properly. Here the
only thing that changes is the training size, which is what lets you say what the
penalty did. One thing at a time.

In [ ]:
# YOUR CODE HERE

## Part 2 · Units, scaling, and what the lasso keeps (20 points)

**(i) Change the units of one column.** Use split `random_state=764` and its first 300
training rows. Fit `Lasso(alpha=300, max_iter=20000)` **four** ways: `Lot_Area` in
square feet or in acres, with or without the scaler.

```python
acres = X.copy()
acres["Lot_Area"] = acres["Lot_Area"] / 43560
```

Split `acres` with the same `random_state` and you get the same houses. For each of the
four fits, report how many coefficients are nonzero, the coefficient on `Lot_Area`, and
held-out MAE. A fitted pipeline knows its own column names:

```python
coef = pd.Series(fit.named_steps["model"].coef_,
                 index=fit.named_steps["prep"].get_feature_names_out())
coef["num__Lot_Area"]
```

**(ii) Ask the lasso twenty times.** Draw twenty different samples of 300 houses from
the full data and fit the **scaled** lasso to each:

```python
for s in range(20):
    rows = np.random.default_rng(s).choice(len(ames), 300, replace=False)
    fit = wide_pipeline(Lasso(alpha=300, max_iter=20000)).fit(X.iloc[rows], y.iloc[rows])
    ...    # record which of the NUMERIC columns got a nonzero coefficient
```

Report, for each of the numeric columns, **the fraction of the twenty fits that kept
it**, as one sorted table. How many were kept every time? Never? Somewhere in between?

⚠ Then look hard at the columns that were **rarely** kept. Are any of them exactly
computable from other columns in the data? `Total_Bsmt_SF` and `Gr_Liv_Area` are good
places to start. **Check it with a line of arithmetic** — do not assert it.

In [ ]:
# YOUR CODE HERE

## Part 3 · The number you tuned on (15 points)

**(i) Three numbers for one tuned model.** Split `random_state=764` again, first 300
training rows. Now choose alpha properly — **inside** resampling — for ridge
(`np.logspace(-1, 4, 13)`) and for lasso (`np.logspace(1, 4, 13)`, with
`max_iter=20000`):

```python
search = GridSearchCV(wide_pipeline(Ridge()), {"model__alpha": grid},
                      cv=KFold(5, shuffle=True, random_state=764))
```

For each model, report the alpha it chose and three numbers:

| | what it is |
|---|---|
| `search.best_score_` | the CV score of the alpha you **chose** — the number you tuned on |
| **nested CV** | `cross_val_score(search, X_300, y_300, cv=KFold(5, shuffle=True, random_state=1))` — the whole search, repeated inside each outer fold. Report the mean **and** the standard deviation of the five scores |
| **held-out** | fit the search on the 300 rows, then `search.score(X_test, y_test)` — **once** |

⚠ Touch the test set once, after choosing. If you look at it in order to choose, it
stops being a held-out number.

⚠ This lab keeps **all 2,930 houses**, including the five over 4,000 square feet that
Meeting 8 sets aside. Leave them in. Whether they matter here is yours to find out.

⏱ Up to a minute: it fits the pipeline almost eight hundred times.

**(ii) Search over nothing but luck.** One small model — the five numeric columns from
Meeting 5 — on 150 houses. Nothing about the model changes. The only thing you vary is
**how the rows are dealt into folds**:

```python
FIVE = ["Gr_Liv_Area", "Year_Built", "Overall_Qual", "Total_Bsmt_SF", "Lot_Area"]
small = Pipeline([("fill", SimpleImputer(strategy="median")),
                  ("scale", StandardScaler()),
                  ("model", LinearRegression())])

for seed in range(20):
    Xa, Xb, ya, yb = train_test_split(ames[FIVE], y, test_size=733, random_state=seed)
    Xa, ya = Xa.iloc[:150], ya.iloc[:150]
    scores = [cross_val_score(small, Xa, ya,
                              cv=KFold(5, shuffle=True, random_state=r)).mean()
              for r in range(100)]
    held_out = small.fit(Xa, ya).score(Xb, yb)
    ...    # for K in (1, 5, 20, 100): the best of the first K scores
```

Report one table. For K = 1, 5, 20 and 100, the median over the twenty splits of:

- **the best CV score among the first K**
- **what the search added** — that best score minus the K = 1 score *on the same split*
- the gap between that best score and the split's held-out score

and, once, the median held-out score. Notice what the held-out score does as K grows.

⚠ This is what *"I tried a few `random_state` values and kept the one that looked
right"* does. Nobody does it on purpose. Almost everybody has done it by accident.

⏱ Up to a minute and a half: twenty splits × 100 deals × 5 folds is ten thousand small
fits.

In [ ]:
# YOUR CODE HERE

## Part 4 · Written (25 points)

Three or four sentences each, and **every answer except (d) points at a number in your
own output.** Padding will not help you. Under each question, *A full answer* says what
earns full credit — the same lines are on the Canvas rubric.

**(a)** From Part 1: at which training size did the penalty earn its keep, and how
much — at the median **and** on the worst split? Then use OLS's R-squared on its own
300 training houses, and the number of columns it had to estimate, to say **why**.

*A full answer:* names the training size; quotes the difference at the median and on the
worst split from your Part 1 table; and uses OLS's training R-squared and its column count
to explain why.

**(b)** From Part 2, two things. First: why did changing the units of `Lot_Area` change
what the **unscaled** lasso kept, and why did the scaled lasso not notice? Second: one
of your rarely-kept columns is exactly a sum or difference of others. Name it, give the
identity, and say what *"the lasso kept `Total_Bsmt_SF` and dropped `Bsmt_Unf_SF`"*
does and does not tell you about houses.

*A full answer:* explains why the unscaled lasso changed and the scaled one did not, from
your four-fit table; names the rarely-kept column and writes the identity you checked; and
says what that selection does and does not tell you.

**(c)** From Part 3. In (i), `best_score_` and the held-out number disagree. **Is that
disagreement the cost of choosing alpha?** Use your nested number to decide: nested CV
repeats the choosing inside every fold, so it pays that cost and `best_score_` does
not. Then (ii): how much did searching add at K = 100, while the model — and its
held-out score — did not change at all? **What is different about the two searches**,
and does it explain the difference in what they cost? Last: which of the three numbers
in (i) would you report, and why?

*A full answer:* uses your nested number to decide whether the (i) disagreement is the cost
of choosing alpha; quotes what searching added at K = 100; says what differs between the
two searches; and names the number from (i) you would report, with a reason.

**(d)** 📗 **From the reading.** ISLP §6.2.2 explains, with a picture, why the lasso
sets some coefficients **exactly** to zero while ridge regression shrinks them but
essentially never does. Give that explanation: what shape is each constraint region,
and where does a contour of the residual sum of squares first touch it? Name the figure.

*A full answer:* the shape of each constraint region; where a contour first touches each,
and why that gives zeros for one and not the other; and the figure. No number needed.

⚠ Answer (d) from the reading. It is there, and it is short.

**(e)** A client asks you two things: *which features drive the price*, and *will your
tuned model be as accurate on next year's sales as your cross-validation says?* Write
the two sentences you would put at the top of your report — one per question — each
naming a number from this lab.

*A full answer:* two sentences, one per question, each naming a number from this lab.

### Your answers

*(double-click to edit)*

**(a)**

**(b)**

**(c)**

**(d)**

**(e)**

---

## Before you submit

- [ ] **Run → Restart Kernel and Run All Cells**
- [ ] Part 1 reports medians **and** worst splits, at both training sizes
- [ ] Part 2 has the four-fit units table **and** the twenty-fit selection table
- [ ] Part 3 has three numbers per model in (i) and the K table in (ii)
- [ ] All five written answers are filled in, each pointing at a number
- [ ] Uploaded the `.ipynb` to Canvas

**75 points.**